# 进阶 Kernel 导读：DeepGEMM 与 FlashAttention-3

> G 讲清了 FlashAttention 的 online softmax + tiling 两件套，把「为什么 tiling 能省显存」这个问题回答到了数学层面。但真要把 FP8 GEMM 跑到 H100 理论算力的 70%，光懂算法不够，还得看一份真实的生产级 kernel 源码是怎么把 TMA、WGMMA、warp specialization 拼到一起的。
>
> 这个附录不重写任何 kernel。它是一份阅读地图：按顺序告诉读者该打开哪些源码文件、每个文件负责什么、关键函数在哪里。覆盖对象是当前工业界最有代表性的三类进阶 kernel——DeepSeek 的 FP8 GEMM 库 DeepGEMM、FlashAttention-2/3、以及 Triton 实现的 grouped GEMM。

进阶 kernel 之所以「进阶」，不在于算法多复杂，而在于它要在三个约束之间同时找平衡：硬件指令的特性（比如 Hopper 的 TMA 和 WGMMA 是异步的）、SRAM 的容量上限（一个 block 通常只有 100-200 KB 可用）、以及数值精度（FP8 的动态范围窄到必须 per-block scaling）。算法层面的 online softmax、tiling 这些 trick 在 G 已经建好直觉，本附录的任务是把直觉对接到真实代码里出现的变量名、函数签名和文件路径。

先约定一些术语。**TMA（Tensor Memory Accelerator）**是 Hopper 引入的硬件 DMA 单元，能在 SM 不参与的情况下把一块张量从 HBM 异步搬到 shared memory；**WGMMA（Warp-Group Matrix Multiply Accumulate）**是 Hopper 的异步 tensor core 指令，一条指令完成一个 64×N×16 的矩阵乘加；**warp specialization** 是 CUTLASS 4.x 引入的编程模型，把一个 CTA 内的 warp 分成 producer（DMA，负责搬数据）和 consumer（MMA，负责算）两组，用 `mbarrier` 做同步。这三个词后面会反复出现。

本附录不实测。H100 上跑一次 FP8 kernel 需要 Hopper 架构、CUTLASS 主线、CUDA 12.3+ 的组合，这些不是教学环境能稳定提供的。我们的目标是让读者拿到源码能看懂、能定位关键代码段、能判断哪些设计对应论文里的哪句话。

## 1. 为什么要专门讲进阶 kernel

前面几节的实现都用 PyTorch + Triton 的「教学级」抽象，写出来的代码能跑对，但离生产级性能差一个数量级。差距主要在三件事上。

第一，**PyTorch 的 eager 模式不感知硬件**。`torch.matmul(a, b)` 在 H100 上可能调到 cuBLAS 的 FP8 kernel，也可能退回 BF16 路径，取决于 dtype、shape、alignment 一堆条件。要榨干硬件，就得自己写 kernel，或者用一个能 JIT 出专用 kernel 的库。

第二，**FP8 这类低精度不是「直接把 dtype 改掉」就能用**。FP8 的 E4M3 格式只有 4 位指数，动态范围大约是 $\pm 448$，比 BF16 小一个量级。直接把 BF16 权重 cast 成 FP8，很大概率溢出或下溢。生产级的做法是 per-block scaling：把矩阵切成 128×128 的块，每块单独算一个 scale factor，把数据压到 FP8 能表示的范围内。这套机制是 DeepGEMM 这类库的核心工作。

第三，**MoE 的 grouped GEMM 不能复用普通 GEMM kernel**。dense 模型一次 GEMM 处理一个完整矩阵；MoE 一次 forward 要对 $E$ 个 expert 各算一次 GEMM，每个 expert 的 token 数还不一样。如果朴素地循环 $E$ 次，kernel launch 开销会吃掉一半时间。Grouped GEMM 把 $E$ 个 GEMM 打包成一次 kernel launch，让一个 CTA 处理一个 expert 的一块——这件事 dense GEMM kernel 做不到。

这三件事就是本附录导读的三类 kernel 的来由：DeepGEMM 解决 FP8 + per-block scaling，FlashAttention-3 解决异步指令的 overlap，grouped GEMM 解决 MoE 的批量小矩阵。

## 2. DeepGEMM 源码导读

DeepGEMM 是 DeepSeek 在 2025 年 2 月开源的 FP8 GEMM 库，仓库地址 [deepseek-ai/DeepGEMM](https://github.com/deepseek-ai/DeepGEMM)。它的设计目标非常窄：只做 Hopper 架构上的 FP8 dense 和 grouped GEMM，不追求跨架构兼容。这种窄范围让它能做到极简——核心代码不到 3000 行 CUDA，而同等功能的 CUTLASS Epilogue 部分就超过 20000 行。

DeepGEMM 的两个标志性设计是：**per-block scaling factor 的 TMA 友好 layout**，和 **BF16 输出 + FP32 累加的混合精度路径**。前者让 FP8 数据的量化误差可控，后者保证累加精度不丢。

### 2.1 per-block scaling 的 TMA layout

FP8 GEMM 的标准做法是把矩阵分成 $B_m \times B_n$ 的块（典型 $128 \times 128$），每块算一个 scale factor $s_{ij}$。计算时 $C_{ij} = \sum_k (A_{ik} \cdot s^A_{ik}) (B_{kj} \cdot s^B_{kj})$，输出再乘上对应的输出 scale。

这套机制的工程难点在于：scale factor 本身也是张量，要和数据一起加载进 SRAM。如果 scale 的 layout 不对齐数据 block 的边界，TMA 搬数据时就要多做一次 gather，性能直接掉一半。

DeepGEMM 的解法是让 scale factor 沿 K 维度连续存放，和数据 block 的 K 分块严格对齐。这样 TMA 描述符可以用一个指针 + 两个 stride 就描述完整个 scale 张量，不需要复杂的索引计算。源码里这套 layout 叫 `NVTE_LAYOUT_BLOCK_SCALING`，本质就是一个三维张量 $[M / B_m, K / B_k, 1]$，每行是一个 scale 值。

这种设计还有一个副作用：scale factor 的内存占用极小。一个 $4096 \times 4096$ 的矩阵，$B_m = B_k = 128$，scale 张量只有 $32 \times 32 = 1024$ 个 FP32 值，4 KB，几乎可以忽略。这正是 per-block scaling 相比 per-tensor scaling 的优势——精度提升一个量级，内存开销几乎为零。

### 2.2 BF16 累加与 FP8 的精度平衡

FP8 tensor core 的累加器是 FP32（这是硬件规定的，不能改）。但最终输出往往要存回 BF16——因为下游的 attention、FFN 都以 BF16 为基本 dtype。这中间存在一个精度选择：累加完 FP32 之后，是直接 truncate 到 BF16，还是 round？

DeepGEMM 选择了 stochastically rounded BF16 输出。源码里的 epilogue 函数会调用 `__nv_cvt_fp32_to_bf16` 的随机舍入版本，让大量累加结果的统计期望无偏。这一点在大模型训练里很关键——如果用 truncation，前向误差会朝一个方向偏，几千层累加下来可能把模型训偏。

累加过程本身也有讲究。WGMMA 指令的累加器在寄存器里是 FP32，但跨 K 维度的多次 WGMMA 调用之间，累加结果会留在同一组寄存器里。如果 K 太长（比如 $K = 8192$），累加值会越来越大，到后面的乘加实际上是在「大数 + 小数」，小数的精度被吃掉。DeepGEMM 的做法是定期把累加器「归一化」一次——把当前部分和存到一个 FP32 的「外部累加器」里，然后清零内部累加器，从 0 开始累加下一段。这种内外两层累加器的结构是 CUTLASS Hopper GHMA kernel 的通用模式。

### 2.3 kernel launch 的 grid/block 设计

DeepGEMM 的 GEMM kernel 用经典的 2D grid：`grid = (M / TM, N / TN)`，每个 CTA 负责输出的一块 $C_{ij}$，大小为 $T_M \times T_N$（典型 $128 \times 256$）。CTA 内部用 4 个 warp，分两组：

- **producer warp group**（1 个 warp）：跑 TMA load 指令，把下一块的 A、B 数据从 HBM 搬到 shared memory
- **consumer warp group**（3 个 warp）：跑 WGMMA 指令，对 shared memory 里的数据做矩阵乘加，结果写到 register file

两组之间用 `mbarrier`（Hopper 的异步 barrier）做同步。producer 搬完一块就 signal barrier，consumer 看到 barrier complete 就开始算。这种 producer/consumer 分工就是 warp specialization 的核心。

block 的选择是性能调优的关键。$T_M \times T_N \times T_K$ 决定 SRAM 占用（每个 stage 需要 $T_M \cdot T_K + T_K \cdot T_N$ 个 FP8 byte）、寄存器占用（累加器 $T_M \cdot T_N$ 个 FP32）、以及 pipeline 的 stage 数。DeepGEMM 默认用 4 stage 的 pipeline——producer 在算第 $i$ 块时，consumer 在算第 $i-1$ 块，同时 producer 已经开始为第 $i+1$ 块做 TMA load。这种多 stage 的 overlap 是 Hopper 上把 FP8 GEMM 跑到峰值算力 70%+ 的必要条件。

### 2.4 源码阅读路线

DeepGEMM 的代码组织比 CUTLASS 简单很多，按下面的顺序读能在一两天内读完整条调用链。所有路径相对于仓库根目录。

**入口层（Python）：**

- `deep_gemm/__init__.py` — Python 包入口，导出 `fp8_gemm`、`m_grouped_fp8_gemm` 等顶层函数
- `deep_gemm/dispatcher.py` — 根据 (M, N, K) shape 和 dtype 选择最合适的 JIT 模板
- `deep_gemm/jit_kernels/gemm.py` — JIT 编译入口，把 shape constants 注入 CUDA 模板再调用 NVRTC 编译

**host 侧启动（C++/CUDA）：**

- `csrc/deep_gemm.cuh` — C++ 侧的 launch 入口，设置 grid/block、调用 `cudaLaunchKernel`
- `csrc/utils.cuh` — TMA descriptor 的构造工具，把 PyTorch tensor 转成 `CUtensorMap`

**device kernel（CUDA）：**

- `csrc/includes/kernels/gemm/kernel.cuh` — 主 kernel 模板，定义 producer/consumer 两个函数
- `csrc/includes/kernels/gemm/warp_specializer.cuh` — warp 分工策略，定义 mbarrier 的初始化和 wait
- `csrc/includes/kernels/gemm/epilogue.cuh` — 输出阶段，FP32 累加器转 BF16 + 应用 per-block scale
- `csrc/includes/utils/matmul.cuh` — WGMMA 指令的封装，把 PTX inline asm 包成 C++ 函数

**grouped GEMM 变体：**

- `csrc/includes/kernels/grouped_gemm/kernel.cuh` — MoE 用的 grouped 版本，每个 CTA 处理一个 expert 的一块
- `deep_gemm/jit_kernels/grouped_gemm.py` — grouped 版本的 JIT 入口，处理变长 expert 的 stride 计算

建议的阅读顺序：先读 `__init__.py` 看公开 API 形状，再跳到 `dispatcher.py` 理解 shape 到 kernel 的映射，最后进 `kernel.cuh` 看 producer/consumer 的循环结构。`warp_specializer.cuh` 和 `matmul.cuh` 是细节层，可以最后读。

In [ ]:
# 把 DeepGEMM 的阅读路线打印成可对照的清单
deepgemm_reading_path = [
    ("入口层（Python）", [
        ("deep_gemm/__init__.py",            "包入口，导出 fp8_gemm 等顶层函数"),
        ("deep_gemm/dispatcher.py",          "按 (M,N,K) shape 选 JIT 模板"),
        ("deep_gemm/jit_kernels/gemm.py",    "把 constants 注入 CUDA 模板再 NVRTC 编译"),
    ]),
    ("host 启动（C++/CUDA）", [
        ("csrc/deep_gemm.cuh",   "launch 入口，设 grid/block 启动 kernel"),
        ("csrc/utils.cuh",       "构造 CUtensorMap，把 PyTorch tensor 转成 TMA 描述符"),
    ]),
    ("device kernel（CUDA）", [
        ("csrc/includes/kernels/gemm/kernel.cuh",          "主 kernel，producer/consumer 两段循环"),
        ("csrc/includes/kernels/gemm/warp_specializer.cuh", "warp 分工 + mbarrier 同步"),
        ("csrc/includes/kernels/gemm/epilogue.cuh",        "FP32 → BF16 + per-block scale 应用"),
        ("csrc/includes/utils/matmul.cuh",                 "WGMMA 指令的 PTX inline asm 封装"),
    ]),
    ("grouped GEMM（MoE 变体）", [
        ("csrc/includes/kernels/grouped_gemm/kernel.cuh",  "每 CTA 处理一个 expert 的一块"),
        ("deep_gemm/jit_kernels/grouped_gemm.py",          "变长 expert 的 stride 计算与 JIT"),
    ]),
]

print("DeepGEMM 源码阅读路线")
print("=" * 70)
for layer, files in deepgemm_reading_path:
    print(f"\n[{layer}]")
    for path, desc in files:
        print(f"  {path}")
        print(f"      → {desc}")
print()
print("关键观察：从 __init__.py 一路到 kernel.cuh，")
print("        每一层都只做一件事——选模板、注入常数、启动、搬数据、算。")

## 3. FlashAttention-2/3 的实现细节

G 讲了 FlashAttention 的算法（online softmax + tiling），这个附录关注 FA-2 和 FA-3 在算法之外做了哪些工程优化。算法本身在 FA-1 已经定型，后续两代的改进集中在「怎么让 GPU 更忙」这件事上。

源码位于 [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention)，主要分 CUDA 实现（`csrc/flash_attn/`）和 Triton 实现（`flash_attn/flash_attn_triton.py`）两条路径。生产环境用 CUDA 版本拿性能，Triton 版本用于教学和快速实验。

### 3.1 FlashAttention-2：两步改进

FA-2（Dao 2023）相对 FA-1 的改进可以用两句话概括：**更少的非 matmul FLOPs，更好的并行化**。

第一点源于一个硬件事实：GPU 上 tensor core 的吞吐远高于其他运算（A100 上 BF16 matmul 是 312 TFLOPs，但 element-wise 加法、softmax 这些只有几十 TFLOPs）。FA-1 的 inner loop 里有一些非 matmul 操作（比如 rescale 时要算 $\exp(m_\text{old} - m_\text{new})$），这些操作虽然不密集，但会拖慢整体吞吐。FA-2 把 rescale 操作推迟到外层循环，让 inner loop 几乎只剩 `Q @ K.T` 和 `P @ V` 两个 matmul。这样 tensor core 的利用率从 FA-1 的 40% 左右提升到 70%+。

第二点是并行化策略的调整。FA-1 主要在 batch 和 head 两个维度并行——每个 (batch, head) 组合分到一个 CTA。这在 batch=1 的推理场景下利用率很低，因为 batch=1 意味着只有 num_heads 个 CTA，远不够填满 A100 的 108 个 SM。FA-2 把序列维度也拆开并行：不同的 Q 块（外层循环的 i）分到不同 CTA。这样 batch=1、seq_len=8192 时仍然能产生足够多的 CTA 把 SM 填满。代价是每个 CTA 要独立完成整个 K/V 循环，无法共享 K/V 的加载——但 attention 本身就是每个 query 独立处理所有 key，所以没有额外计算。

### 3.2 FlashAttention-3：Hopper 专属的三个特性

FA-3（Shah et al. 2024）是 Hopper 专属实现，相对 FA-2 的提升来自对三个新硬件特性的利用。

**asynchronous softmax（WGMMA + softmax overlap）**。FA-2 的 inner loop 是串行的：先算 $QK^T$，再算 softmax，再算 $PV$。三步用不同的硬件资源——matmul 用 tensor core，softmax 用 CUDA core。FA-3 用 WGMMA 的异步特性，让下一块的 matmul 和当前块的 softmax 在同一个 warp group 里 overlap。具体做法是把每个 warp 的执行分成「issue WGMMA → 等 WGMMA 完成 → 算 softmax」三段，配合 ping-pong scheduling 让两组寄存器交替使用，一个 warp 始终在算。

**ping-pong scheduling**。这是 FA-3 性能提升的关键 trick。Hopper 的 WGMMA 是异步的——发起指令后 warp 可以继续做别的事，过一段时间再来 check 结果。FA-3 用两个 accumulator buffer（A 和 B），在算第 $i$ 块时：发起第 $i$ 块的 WGMMA → 立刻处理第 $i-1$ 块的 softmax（用 buffer A）→ check 第 $i$ 块 WGMMA 完成 → 发起第 $i+1$ 块 WGMMA → 处理第 $i$ 块的 softmax（用 buffer B）→ ...。这种 A/B 交替让 matmul 和 softmax 几乎完全 overlap，非 matmul 开销被隐藏到接近零。

**FP8 tensor core 利用**。FA-3 的 forward 可以把 $QK^T$ 和 $PV$ 切到 FP8（E4M3）tensor core，吞吐是 BF16 的两倍。但 softmax 部分仍用 FP32 累加保证数值稳定——online softmax 的 $m$、$d$、$o$ 都是 FP32，scale factor 是 per-head 的。这种「FP8 matmul + FP32 accumulate」的混合策略和 DeepGEMM 的 per-block scaling 思路一脉相承。

### 3.3 FlashAttention 源码阅读路线

FlashAttention 仓库代码量比 DeepGEMM 大一个量级，建议按下面顺序切入。所有路径相对于 [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) 仓库根目录。

**Python 入口：**

- `flash_attn/flash_attn_func.py` — 公开 API，单次 attention 调用
- `flash_attn/flash_attn_varlen_func.py` — 变长版本，MoE 和 packing 场景用

**CUDA 实现（FA-2 + FA-3）：**

- `csrc/flash_attn/flash_api.cpp` — PyTorch C++ extension 入口，参数校验和 launch
- `csrc/flash_attn/flash_fwd_kernel.h` — forward kernel 的核心模板，分 causal / non-causal 两套
- `csrc/flash_attn/flash_fwd_launch.h` — launch 配置，按 (seqlen, headdim, dtype) 选 grid/block
- `csrc/flash_attn/kernels_utils.h` — softmax、rescale、scale 的 device 函数
- `csrc/flash_attn epilogue/epilogue_fwd.hpp` — forward 输出阶段的 $o / d$ 归一化

**FA-3 专属（Hopper，在 `hopper` 子目录）：**

- `csrc/flash_attn/hopper/flash_fwd_kernel.h` — FA-3 主 kernel，含 ping-pong scheduling
- `csrc/flash_attn/hopper/epilogue_fwd.hpp` — FA-3 epilogue，处理 FP8 scale
- `csrc/flash_attn/hopper/utils.h` — TMA descriptor 构造和 mbarrier 同步

**Triton 实现（教学友好）：**

- `flash_attn/flash_attn_triton.py` — 完整 FlashAttention 的 Triton 版本，逐行可读

建议路径：先读 `flash_attn_triton.py` 建立对 tiling 循环的直觉（300 行 Python），再对照 `csrc/flash_attn/flash_fwd_kernel.h` 看工程优化。FA-3 部分单独看 `hopper/` 子目录，那里是 ping-pong scheduling 和 WGMMA overlap 的实现。

In [ ]:
# FlashAttention 源码阅读路线
fa_reading_path = [
    ("Python 入口", [
        ("flash_attn/flash_attn_func.py",      "单次 attention 调用的公开 API"),
        ("flash_attn/flash_attn_varlen_func.py", "变长版本，MoE / packing 用"),
    ]),
    ("CUDA 实现（FA-2 + FA-3 共用）", [
        ("csrc/flash_attn/flash_api.cpp",           "PyTorch ext 入口，参数校验 + launch"),
        ("csrc/flash_attn/flash_fwd_kernel.h",      "forward kernel 模板"),
        ("csrc/flash_attn/flash_fwd_launch.h",      "按 shape 选 grid/block 的 launch 配置"),
        ("csrc/flash_attn/kernels_utils.h",         "softmax / rescale 的 device 函数"),
    ]),
    ("FA-3 Hopper 专属", [
        ("csrc/flash_attn/hopper/flash_fwd_kernel.h", "ping-pong scheduling 主 kernel"),
        ("csrc/flash_attn/hopper/epilogue_fwd.hpp",   "FP8 scale 处理的 epilogue"),
        ("csrc/flash_attn/hopper/utils.h",            "TMA descriptor + mbarrier 同步"),
    ]),
    ("Triton 实现（教学友好）", [
        ("flash_attn/flash_attn_triton.py", "完整 FA 的 Triton 版本，约 300 行"),
    ]),
]

print("FlashAttention 源码阅读路线")
print("=" * 70)
for layer, files in fa_reading_path:
    print(f"\n[{layer}]")
    for path, desc in files:
        print(f"  {path}")
        print(f"      → {desc}")
print()
print("关键观察：先读 Triton 版建立直觉，再对照 CUDA 版看工程优化。")
print("        FA-3 的 hopper/ 子目录是学习 ping-pong scheduling 的最佳教材。")

## 4. Triton 基础：看懂 block-level programming

Triton 是 OpenAI 维护的 GPU kernel DSL，语法接近 Python，但底层生成的是和 CUDA 一样高效的 PTX。它的核心抽象是 **block-level programming**——程序员写的是「一个 block 的逻辑」，Triton 编译器负责把这个逻辑映射到 SM 上的 warp 和 lane。

这一节不要求读者自己写 Triton。目标是读别人的 Triton kernel 时能看懂——比如 FA 的 Triton 实现、vLLM 的自定义 op、xformers 的 fused kernel。

### 4.1 三个核心 op：tl.load / tl.store / tl.dot

Triton 的程序结构是：一个 `@triton.jit` 装饰的函数，参数里有一组「指针」和一组「shape」；函数体里用 `tl.load` 把数据从 HBM 读到 block 寄存器，用 `tl.dot` 做 block 级别的矩阵乘，用 `tl.store` 把结果写回 HBM。

**`tl.load(ptr, mask, other)`**。`ptr` 是一个 block 的指针（通常是 `ptr + offset`，其中 offset 是一个 `[BLOCK_SIZE]` 的 arange）；`mask` 是一个 `[BLOCK_SIZE]` 的 bool 张量，用来处理边界（比如序列末尾不足一个 block）；`other` 是 mask 为 False 时的填充值（常填 0 或 -inf）。这条指令在底层会展开成一组 global memory load + shared memory write，本质就是 CUDA 里的 `cudaMemcpyAsync`。

**`tl.dot(a, b)`**。`a` 是 `[M, K]`，`b` 是 `[K, N]`，返回 `[M, N]`。这是 Triton 唯一会调到 tensor core 的 op——其他所有运算（加、乘、exp）都用 CUDA core。这一点解释了为什么 FlashAttention 的实现要把算法重写成「mostly matmul」的形式：只有把 rescale 和 softmax 的计算量压到最小，tensor core 才能跑满。

**`tl.store(ptr, value, mask)`**。和 `tl.load` 对称，把 block 寄存器的值写回 HBM。`mask` 同样处理边界。

除了这三个，还有 `tl.arange`、`tl.exp`、`tl.maximum` 这些 element-wise op，语义和 PyTorch 类似，但作用对象是 block 寄存器（一个 `[BLOCK_SIZE]` 的 SIMD 向量），不是单个标量。

### 4.2 一个最小的 Triton matmul

下面是 Triton 官方教程里的最小 matmul（去掉了 autotune 和 mask 简化版）。它示范了 block-level programming 的所有核心模式：外层 grid 循环、内层 K 分块循环、load/dot/store 三件套。整段不到 30 行。

```python
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,   # 三个张量的 HBM 基地址
    M, N, K,               # 矩阵的 shape（标量）
    stride_am, stride_ak,  # A 的行 / 列 stride
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, # block 大小（编译期常量）
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    # 每个 CTA 负责输出的一块 C[pid_m, pid_n]
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # 算这个 block 内的行/列索引
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    # A 和 B 的初始指针（K 维度的起点）
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    # 累加器，FP32
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # K 维度的分块循环
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs)         # [BLOCK_M, BLOCK_K]
        b = tl.load(b_ptrs)         # [BLOCK_K, BLOCK_N]
        acc = tl.dot(a, b, acc)     # tensor core matmul
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # 写回 C
    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(c_ptrs, acc)
```

逐行解读这段代码能看到所有 Triton 的核心模式：

- `tl.program_id(0)` 和 `tl.program_id(1)` 是这个 CTA 在 grid 里的坐标，等价于 CUDA 里的 `blockIdx.x` / `blockIdx.y`
- `tl.arange(0, BLOCK_M)` 生成 `[BLOCK_M]` 的 arange，和 `offs_m` 做广播得到 `[BLOCK_M, BLOCK_K]` 的索引矩阵
- `a_ptrs` 不是数据本身，而是「下一次 `tl.load` 要从哪里读」的指针矩阵
- `tl.constexpr` 标注的 block 大小是编译期常量，Triton 会针对每个 (BLOCK_M, BLOCK_N, BLOCK_K) 组合生成一份专门的 PTX
- `tl.dot(a, b, acc)` 是累加形式的 matmul，等价于 `acc += a @ b`，但内部直接调 tensor core 指令

把这段 matmul 和 G 的 PyTorch FlashAttention 实现对照读，会发现结构几乎相同——都是「外层 Q 块循环、内层 K/V 块循环、每块 load → 计算 → 更新 running 变量」。Triton 的角色是把这种 Python 写的 block-level 逻辑翻译成高效的 PTX，让算法工程师不用直接写 CUDA。

## 5. MoE 专属 kernel：grouped GEMM 与 scatter/gather

MoE 模型的 forward 路径里，attention 部分和 dense 模型一样，可以用标准 FlashAttention。真正的麻烦在 FFN 部分：每个 token 被路由到 $k$ 个 expert，每个 expert 是一个独立的 FFN，要对分配到自己的 token 子集做一次 GEMM。如果朴素地循环 $E$ 个 expert，每个 expert 调一次 cuBLAS GEMM，性能会很差——因为每个 expert 的 token 数通常很少（64-512），小矩阵 GEMM 的算力利用率不到 10%。

Grouped GEMM 是为这个场景设计的专用 kernel。

### 5.1 grouped GEMM：一次 launch 处理所有 expert

Grouped GEMM 的输入是 $E$ 组 $(A_e, B_e)$，输出是 $E$ 个 $C_e = A_e B_e$。一次 kernel launch 把所有 expert 都算完。关键设计在于：把 $E$ 个独立的 GEMM 看作一个「虚拟的大 GEMM」，CTA 的 grid 在 (M, N) 之外多加一个维度 E。

具体实现有两种变体。第一种是 **padding 模式**：所有 expert 的 token 数 padding 到同一个值 `max_tokens`，grid 变成 `(E, max_tokens / BLOCK_M, N / BLOCK_N)`。每个 CTA 知道自己在哪个 expert、哪块 M、哪块 N，从对应的 `A[e]`、`B[e]` 读数据。这种实现简单但浪费——padding 的部分在做无用功。G 的 9.2 节详细算过这种浪费。

第二种是 **contiguous / varlen 模式**：把所有 expert 的有效 token 拼成一个长 1D 张量，配合一个 `cu_seqlens` 数组记录每个 expert 的起止位置。CTA 的 grid 是 `(total_tokens / BLOCK_M, N / BLOCK_N)`，每个 CTA 通过 `cu_seqlens` 二分查找出自己属于哪个 expert。这种实现没有 padding 浪费，是 DeepGEMM 和 FlashAttention varlen 的标准做法。

DeepGEMM 的 grouped GEMM 实现位于 `csrc/includes/kernels/grouped_gemm/kernel.cuh`，和它的 dense GEMM 共用同一套 warp specialization 和 epilogue 基础设施。差异只在于：grid 多了一个 expert 维度，TMA descriptor 要按 expert 分别构造，scale factor 也按 expert 分别存。

### 5.2 Triton 的 grouped GEMM 代码片段

下面是 Triton 实现的 grouped GEMM 的简化版本（来自 Triton 官方 tutorial 的 `09-persistent-matmul` 风格改编）。和 4.2 的普通 matmul 相比，多了两件事：per-expert 的指针偏移、以及 `cu_seqlens` 的二分查找。

```python
@triton.jit
def grouped_matmul_kernel(
    a_ptr, b_ptr, c_ptr,      # 所有 expert 拼接后的 1D 张量
    cu_seqlens_ptr,           # [E+1]，每个 expert 的累积起止位置
    N, K,                     # 每个 expert 的权重 shape（共享）
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    EXPERTS: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)  # M 维度的 block 编号（全局）
    pid_n = tl.program_id(1)  # N 维度的 block 编号

    # 二分查找 pid_m 落在哪个 expert
    # cu_seqlens = [0, e0_tokens, e0_tokens + e1_tokens, ...]
    expert_id = 0
    for e in range(EXPERTS):
        end = tl.load(cu_seqlens_ptr + e + 1)
        # 如果 pid_m * BLOCK_M < end，说明落在这个 expert
        # Triton 的控制流里这里要写 mask
        expert_id = tl.where(pid_m * BLOCK_M < end, e, expert_id)

    # 算出当前 block 在该 expert 内的 M 偏移
    start = tl.load(cu_seqlens_ptr + expert_id)
    local_m = pid_m * BLOCK_M - start

    # 后面和普通 matmul 完全一样，只是指针要加上 expert 的偏移
    a_ptrs = a_ptr + (local_m[:, None] * stride_am +
                     tl.arange(0, BLOCK_K)[None, :] * stride_ak)
    b_ptrs = b_ptr + (expert_id * N * K +   # 每个 expert 的 B 在 B 大张量里的偏移
                      tl.arange(0, BLOCK_K)[:, None] * stride_bk +
                      tl.arange(0, BLOCK_N)[None, :] * stride_bn)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs)
        b = tl.load(b_ptrs)
        acc = tl.dot(a, b, acc)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = c_ptr + (local_m[:, None] * stride_cm +
                      tl.arange(0, BLOCK_N)[None, :] * stride_cn)
    tl.store(c_ptrs, acc)
```

和 4.2 的普通 matmul 对比，grouped 版本多出的代码量很少——核心就是开头那段 `expert_id` 的查找。这也解释了为什么 grouped GEMM 比逐 expert 循环快：一次 kernel launch 的开销被 $E$ 个 expert 摊销，每个 expert 内部的 GEMM 逻辑和 dense 版本完全相同，不需要为「变长」做额外的复杂处理。

### 5.3 expert routing 的 scatter/gather kernel

Grouped GEMM 之外，MoE 还有另一类专用 kernel：scatter / gather。它们处理的是 token 在「原序列位置」和「expert 内部连续位置」之间的搬运。

forward 过程分四步：

1. **router 计算**：对每个 token 算一个 $E$ 维 logits，挑 top-k 个 expert
2. **scatter**（dispatch）：把每个 token 复制到它路由到的 $k$ 个 expert，每个 expert 内部 token 连续存放
3. **grouped GEMM**：所有 expert 并行计算 FFN
4. **gather**（combine）：把每个 expert 的输出按路由权重加权合并回原序列位置

scatter 和 gather 本质是带 mask 的 memory copy——每个 token 要写到 1-2 个不连续的目标位置（expert 内部），读的时候也是从多个 expert 读回来加权。这类 kernel 用 CUDA 写要处理大量边界条件（变长、top-2、expert 不均衡），但用 Triton 写可以非常简洁：核心就是一个 `tl.load` + 一个 `tl.store`，配一个路由表的 mask。

vLLM 和 SGLang 的 MoE 实现里都有这种 scatter/gather kernel，通常和 grouped GEMM 融合成一个 fused MoE kernel——dispatch → GEMM1 → activation → GEMM2 → combine 一次 kernel launch 完成。这种 fusion 能把中间张量（每次 GEMM 的输出）完全留在 SRAM，避免写回 HBM，是 MoE 推理性能的关键。

### 5.4 为什么 MoE kernel 比 dense GEMM 难写

把 dense GEMM 改成 grouped GEMM 看起来只是多了一个 expert 维度，工程上却引入三类新难点。

**第一，变长**。每个 expert 的 token 数不固定——受 router 的动态决策影响，某个 expert 可能收到 10 个 token，另一个收到 1000 个。grouped GEMM 必须支持变长输入，这要求 kernel 内部做边界判断、scale factor 按 expert 分别存、TMA descriptor 按 expert 分别构造。dense GEMM 这些都是常量。

**第二，load imbalance**。如果 router 把 90% 的 token 都路由到一个 expert，那个 expert 的 GEMM 会成为瓶颈，其他 expert 的 SM 全部空转。kernel 层面解决不了这个问题，需要 router 训练时加 auxiliary loss 做负载均衡（第 10 节讲过）。但 kernel 可以做的是：在 launch 时动态调整 grid，让 token 多的 expert 多分一些 CTA。这种 dynamic grid 是 dense GEMM 不需要的。

**第三，fusion 复杂度**。dense 模型的 FFN 是 `Linear → activation → Linear` 三个独立 op，每个 op 之间张量写回 HBM。MoE 的 FFN 把这三步融合进一个 kernel，输出大小是变长的，activation 要在 SRAM 里做——这要求 kernel 同时管理输入 scatter、两次 GEMM 的中间结果、输出 gather 三类张量的生命周期。CUTLASS 的 Hopper MoE 示例（`examples/55_hopper_moe`）有完整实现，是学习 fused MoE kernel 的最佳参考。

In [ ]:
# 用 NumPy 模拟 MoE scatter 的数据流，帮助理解 grouped GEMM 的输入是怎么来的
import numpy as np

np.random.seed(42)

# 模拟场景：8 个 token，3 个 expert，top-1 路由
n_tokens = 8
n_experts = 3
d = 4

# 每个 token 被路由到哪个 expert（router 的输出）
routing = np.random.randint(0, n_experts, size=n_tokens)
print(f"routing 决策: {routing}")
print(f"每个 token 的 hidden state shape: [{n_tokens}, {d}]")

# scatter：按 expert 分组，每个 expert 内部 token 连续存放
cu_seqlens = [0]
for e in range(n_experts):
    cu_seqlens.append(cu_seqlens[-1] + (routing == e).sum())
print(f"\ncu_seqlens (累积起止): {cu_seqlens}")
print("含义: expert 0 占 [0, 3), expert 1 占 [3, 6), expert 2 占 [6, 8)")

# 构造 scatter 后的连续 buffer
scattered = np.zeros((n_tokens, d))
perm = []
for e in range(n_experts):
    for i, r in enumerate(routing):
        if r == e:
            perm.append(i)
perm = np.array(perm)
print(f"\npermute 顺序 (token 原位置 -> 新位置): {perm}")
print("含义: 把原 token [2, 5, 7] 搬到 expert 0 的 [0,1,2] 位置")
print("       原因是 router 把这三个 token 路由到了 expert 0")

print()
print("关键观察：")
print("  scatter 后的 buffer 是连续的，可以直接喂给 grouped GEMM kernel")
print("  cu_seqlens 告诉 kernel 每个 expert 的起止位置")
print("  gather 是 scatter 的逆操作——按 routing 权重合并回原位置")

## 6. 学习路径

想深入 GPU kernel 工程的读者，建议按下面的顺序推进。核心原则是：先建立 block-level 的直觉，再读生产级 kernel，最后才动手写自己的 kernel。直接跳到最后一步会卡死在 CUDA 的细节里。

**第一步：建立硬件直觉**。读 A（GPU 硬件）理解 SM、tensor core、HBM/SRAM 的层级。重点理解 roofline model——一个 op 是 compute-bound 还是 memory-bound，决定了优化方向。这一步不需要写代码。

**第二步：学 Triton**。从 Triton 官方 tutorial 的 `01-vector-add` 开始，按顺序读到 `06-fused-attention`。前 5 个 tutorial 覆盖了所有基础 op（load / store / dot / reduce），第 6 个是 FlashAttention 的简化版本，能看懂它就读懂了 G 的 PyTorch 版本在 GPU 上长什么样。

**第三步：读 FlashAttention 源码**。先读 `flash_attn/flash_attn_triton.py` 的 Triton 版本（约 300 行），再对照 FA-2 的 CUDA 版本（`csrc/flash_attn/flash_fwd_kernel.h`）。目标是理解 FA-2 相对 FA-1 的两点改进在代码里长什么样。

**第四步：读 DeepGEMM 源码**。按本附录 2.4 节的路线读。DeepGEMM 的代码量比 FlashAttention 少很多，但涉及更现代的 Hopper 特性（TMA、WGMMA、warp specialization）。如果 FA 的 CUDA 版本能读懂，DeepGEMM 也能读懂。

**第五步：读 CUTLASS Hopper 示例**。CUTLASS 是 NVIDIA 的 CUDA 模板库，所有现代 GEMM kernel（包括 DeepGEMM）都建立在它的抽象之上。`examples/55_hopper_moe` 是 MoE 的完整示例，`examples/48_hopper_fmha` 是 FlashAttention 的 CUTLASS 实现。读这两份示例能理解 CUTLASS 的 producer/consumer 模型。

**第六步：动手写**。从修改 FlashAttention 的 Triton 版本开始——加一个 causal mask、改一个 block size、加一个 head_dim 分支。每次只改一件事，跑 benchmark 看性能变化。能独立完成这种修改后，再尝试写一个全新的 kernel。

### 6.1 推荐资源

**Talk / 视频：**

- Phil Tillet, [Triton: An Intermediate Language and Compiler for Tiled Neural Network Computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), MAPL 2019 — Triton 的原始论文，解释 block-level programming 的设计动机
- Tri Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691) — 作者本人 talk 在 YouTube 上有多个版本，重点看 work partitioning 部分
- Tri Dao & Brian Karrer, [FlashAttention-3 talk at GPU Mode](https://www.youtube.com/watch?v=iVCQuQmBx7Y) — FA-3 的 Hopper 优化细节，包括 ping-pong scheduling
- [GPU Mode Discord](https://discord.gg/gpumode) — GPU kernel 工程师社区，NVIDIA / OpenAI / Anthropic 的 kernel 工程师都在里面，定期有 talk

**论文阅读顺序：**

1. Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022 — online softmax + tiling 的原始论文
2. Dao, [FlashAttention-2](https://arxiv.org/abs/2307.08691), 2023 — work partitioning 和并行化改进
3. Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024 — Hopper 的 WGMMA / TMA / FP8
4. DeepSeek-AI, [DeepGEMM Technical Report](https://github.com/deepseek-ai/DeepGEMM/blob/main/README.md) — per-block scaling 与 BF16 累加的工程实现
5. Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018 — online softmax 的数学基础

**代码仓库：**

- [deepseek-ai/DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) — FP8 GEMM 库
- [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) — FlashAttention 全家桶
- [openai/triton](https://github.com/openai/triton) — Triton 编译器和 tutorial
- [NVIDIA/cutlass](https://github.com/NVIDIA/cutlass) — CUDA 模板库，Hopper kernel 的基础设施
- [triton-lang/triton](https://triton-lang.org/main/getting-started/tutorials/index.html) — Triton 官方 tutorial

**社区与 benchmark：**

- [GPU Mode Discord](https://discord.gg/gpumode) 的 `#flash-attention`、`#triton`、`#cutlass` 频道
- [Stanford CRFM benchmark suite](https://crfm.stanford.edu/) — 各种 kernel 的实测吞吐对比

## 小结

- [ ] 进阶 kernel 的核心约束是：硬件指令的异步特性（TMA / WGMMA）、SRAM 容量上限、FP8 的低动态范围
- [ ] DeepGEMM 用 per-block scaling 把 FP8 数据压进 E4M3 范围，scale factor 的 layout 与 TMA descriptor 对齐
- [ ] BF16 输出配合 FP32 累加器是 FP8 GEMM 的标配，输出阶段用 stochastically rounded truncate 避免偏差
- [ ] warp specialization 把 CTA 内的 warp 分成 producer（TMA load）和 consumer（WGMMA），用 mbarrier 同步
- [ ] FlashAttention-2 的两步改进：减少非 matmul FLOPs、在 seq 维度并行，让 batch=1 也能填满 SM
- [ ] FlashAttention-3 用 ping-pong scheduling 让 softmax 和 WGMMA overlap，加上 FP8 tensor core 达到 H100 峰值算力 75%
- [ ] Triton 的三个核心 op 是 `tl.load` / `tl.store` / `tl.dot`，其中 `tl.dot` 是唯一调 tensor core 的 op
- [ ] block-level programming 让算法工程师写 block 级逻辑，Triton 编译器负责映射到 SM 上的 warp 和 lane
- [ ] grouped GEMM 把 E 个 expert 的 GEMM 打包成一次 kernel launch，用 `cu_seqlens` 处理变长
- [ ] MoE kernel 难写在三点：变长、load imbalance、fusion 复杂度（scatter → GEMM → activation → GEMM → gather）

## 作业

下面三道作业帮你熟悉 DeepGEMM 和 FlashAttention 的源码结构。可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI「做完这道题」——重要的是自己打开仓库对照文件路径读一遍。

**作业 1：识别 DeepGEMM 文件的职责**

下面列出了 DeepGEMM 仓库里的三个文件路径。请把每个文件和它的职责描述匹配起来。

小提示：参考第 2.4 节的源码阅读路线。

In [ ]:
# 作业 1：DeepGEMM 文件职责匹配
files = [
    "csrc/includes/kernels/gemm/kernel.cuh",
    "csrc/includes/kernels/gemm/epilogue.cuh",
    "csrc/includes/kernels/gemm/warp_specializer.cuh",
]
descriptions = [
    "定义 producer / consumer warp 分工，初始化和等待 mbarrier",
    "主 kernel 模板，包含外层 K 循环和 load / dot / store 的核心逻辑",
    "FP32 累加器转 BF16 输出，并应用 per-block scale factor",
]

# TODO: 把 descriptions 重排，让它和 files 一一对应
# 提示：kernel.cuh 是主循环，epilogue 是输出阶段，warp_specializer 是 warp 分工
answer = None  # TODO: 填一个 list，元素是 descriptions 的下标重排，例如 [1, 2, 0]

assert answer is not None, "请先填入答案"
assert len(answer) == 3, "答案应该有 3 个元素"
assert sorted(answer) == [0, 1, 2], "答案应该是 [0, 1, 2] 的重排"

# 正确匹配：
#   kernel.cuh          → 主循环模板                → descriptions[1]
#   epilogue.cuh        → FP32 → BF16 + scale       → descriptions[2]
#   warp_specializer.cuh → producer/consumer + mbarrier → descriptions[0]
expected = [1, 2, 0]
assert answer == expected, (
    "提示：kernel.cuh 是主循环（含 K 分块循环），"
    "epilogue 是输出阶段（FP32->BF16+scale），"
    "warp_specializer 定义 producer/consumer 的 mbarrier 同步"
)

print("作业 1 通过：")
for f, i in zip(files, answer):
    print(f"  {f}")
    print(f"    → {descriptions[i]}")
print()
print("学会了：DeepGEMM 的 kernel / epilogue / warp_specializer 三件套，")
print("        分别对应主循环、输出、warp 同步——这是 Hopper GEMM kernel 的标准分层。")

**作业 2：识别 FlashAttention-3 的关键文件**

FlashAttention-3 的 Hopper 专属代码在 `csrc/flash_attn/hopper/` 子目录下。下面列出三个文件，请选出**包含 ping-pong scheduling 主 kernel** 的那个。

小提示：参考第 3.3 节。ping-pong scheduling 是 FA-3 性能提升的核心 trick，让 softmax 和 WGMMA overlap。

In [ ]:
# 作业 2：选出 FA-3 的 ping-pong scheduling 主 kernel 文件
candidates = {
    "A": "csrc/flash_attn/hopper/utils.h",
    "B": "csrc/flash_attn/hopper/flash_fwd_kernel.h",
    "C": "csrc/flash_attn/hopper/epilogue_fwd.hpp",
}

print("候选文件：")
for k, v in candidates.items():
    print(f"  ({k}) {v}")
print()

answer = None  # TODO: 填 "A" / "B" / "C"

assert answer in {"A", "B", "C"}, "答案必须是 A / B / C 之一"
assert answer == "B", (
    "提示：ping-pong scheduling 是 forward kernel 的核心逻辑，"
    "        应该在 _kernel_ 文件里，不在 utils 或 epilogue 里"
)

print(f"作业 2 通过：FA-3 的 ping-pong scheduling 主 kernel 是")
print(f"  {candidates[answer]}")
print()
print("学会了：FA-3 的 hopper/ 子目录按 _kernel / epilogue / utils 分层，")
print("        ping-pong scheduling 这种核心算法逻辑永远在 _kernel 文件里。")

**作业 3：识别 grouped GEMM 的关键设计**

下面三句话里，哪一句准确描述了 grouped GEMM 相对 dense GEMM 的核心区别？

小提示：参考第 5.1 节。grouped GEMM 的核心是「把 E 个独立 GEMM 打包成一次 launch」。

In [ ]:
# 作业 3：选出 grouped GEMM 的核心设计
options = {
    "A": (
        "grouped GEMM 把每个 expert 的权重压缩到 INT4，"
        "用更低的精度换更高的吞吐"
    ),
    "B": (
        "grouped GEMM 把 E 个 expert 的 GEMM 打包成一次 kernel launch，"
        "grid 多一个 expert 维度，用 cu_seqlens 处理变长"
    ),
    "C": (
        "grouped GEMM 把所有 expert 的权重拼接成一个大矩阵，"
        "调一次普通的 cuBLAS GEMM 完成计算"
    ),
}

print("候选描述：")
for k, v in options.items():
    print(f"  ({k}) {v}")
print()

answer = None  # TODO: 填 "A" / "B" / "C"

assert answer in {"A", "B", "C"}, "答案必须是 A / B / C 之一"
assert answer == "B", (
    "提示：A 是量化（和 grouped 无关），C 是拼接（仍然一次 dense GEMM，处理不了变长 expert）；"
    "        grouped GEMM 的核心是「E 个 GEMM 一次 launch」+ 「cu_seqlens 处理变长」"
)

print(f"作业 3 通过：grouped GEMM 的核心设计是")
print(f"  {options[answer]}")
print()
print("学会了：grouped GEMM 解决的是「E 个变长小 GEMM」的批量计算问题，")
print("        不是量化、也不是简单的拼接——它需要一个支持变长的专用 kernel。")
print("        DeepGEMM、CUTLASS、FlashAttention varlen 都实现了这个模式。")

## 参考文献

- DeepSeek-AI, [DeepGEMM repository](https://github.com/deepseek-ai/DeepGEMM), 2025
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022
- Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691), 2023
- Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024
- Tillet et al., [Triton: an intermediate language and compiler for tiled neural network computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), MAPL 2019
- Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018
- NVIDIA, [CUTLASS Hopper examples](https://github.com/NVIDIA/cutlass/tree/main/examples), 2024
- [OpenAI Triton tutorials](https://triton-lang.org/main/getting-started/tutorials/index.html)